In [1]:
# imports and env
import base64
from dotenv import load_dotenv
from langchain_unstructured.document_loaders import UnstructuredLoader
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_community.vectorstores.utils import filter_complex_metadata
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate

load_dotenv()

/home/rikesh/Rikesh/RAG/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
# load PDF
PDF_PATH = "../data/crag_paper.pdf"

loader = UnstructuredLoader(
    PDF_PATH,
    mode="elements",
    strategy="hi_res",
    extract_images_in_pdf=True,
)
elements = loader.load()

print(f"Loaded {len(elements)} elements")
for cat in sorted(set(el.metadata.get("category", "unknown") for el in elements)):
    count = sum(1 for el in elements if el.metadata.get("category") == cat)
    print(f"  {cat}: {count}")

INFO: pikepdf C++ to Python logger bridge initialized
INFO: Downloading spaCy model en_core_web_sm 3.8.0 …
INFO: Installed en_core_web_sm 3.8.0
INFO: HTTP Request: HEAD https://huggingface.co/unstructuredio/yolo_x_layout/resolve/main/yolox_l0.05.onnx "HTTP/1.1 302 Found"
INFO: HTTP Request: GET https://huggingface.co/api/models/unstructuredio/yolo_x_layout/xet-read-token/7680d6f857780bcf8d49916aa2e8881bd49dee3e "HTTP/1.1 200 OK"
INFO: Reading PDF for file: ../data/crag_paper.pdf ...


Loaded 255 elements
  FigureCaption: 6
  Footer: 1
  Formula: 1
  Header: 1
  Image: 6
  ListItem: 41
  NarrativeText: 107
  Table: 7
  Title: 32
  UncategorizedText: 53


In [3]:
elements[0].metadata.keys()

dict_keys(['source', 'coordinates', 'last_modified', 'filetype', 'languages', 'page_number', 'file_directory', 'filename', 'category', 'element_id'])

In [4]:
for element in elements:
    if element.metadata.get("category") == "Image":
        print(element.metadata.keys())
        print(element.metadata)
        break

dict_keys(['source', 'coordinates', 'last_modified', 'filetype', 'languages', 'page_number', 'image_path', 'file_directory', 'filename', 'category', 'element_id'])
{'source': '../data/crag_paper.pdf', 'coordinates': {'points': ((np.float64(372.93637515555537), np.float64(2949.874959546666)), (np.float64(372.93637515555537), np.float64(3083.2151141477775)), (np.float64(470.7191490011109), np.float64(3083.2151141477775)), (np.float64(470.7191490011109), np.float64(2949.874959546666))), 'system': 'PixelSpace', 'layout_width': 2894, 'layout_height': 4093}, 'last_modified': '2026-08-14T08:25:21', 'filetype': 'application/pdf', 'languages': ['eng'], 'page_number': 1, 'image_path': '/home/rikesh/Rikesh/RAG/multimodal_rag/notebook/figures/figure-1-1.jpg', 'file_directory': '../data', 'filename': 'crag_paper.pdf', 'category': 'Image', 'element_id': '9c5fdac1264f8a2f2f4ea2b1a0c7cda6'}


In [5]:
# caption images with VLM
vlm = ChatOpenAI(model="gpt-5-mini")

IMAGE_CAPTION_SYSTEM_PROMPT = """You are a document analysis assistant. Your task is to generate \
detailed, accurate descriptions of images extracted from a document. These descriptions will be \
embedded into a vector store and used for semantic retrieval, so they must capture all information \
a user might search for.

For each image, describe:
- The image type (chart, diagram, photograph, table, illustration, screenshot, etc.)
- All visible text, labels, titles, captions, and annotations
- Key data, values, trends, or patterns (especially for charts and graphs)
- The main subject and all important visual elements
- Spatial relationships and structure where relevant

Be specific and thorough. Avoid vague language."""

def encode_image(image_path: str) -> str:
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

def caption_image(image_path: str) -> str:
    b64 = encode_image(image_path)
    messages = [
        SystemMessage(content=IMAGE_CAPTION_SYSTEM_PROMPT),
        HumanMessage(content=[
            {"type": "text", "text": "Describe this image extracted from a document."},
            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{b64}"}},
        ]),
    ]
    return vlm.invoke(messages).content

In [6]:
image_docs = []

for el in elements:
    if el.metadata.get("category") == "Image":
        image_path = el.metadata.get("image_path", "")
        if image_path:
            caption = caption_image(image_path)
            image_docs.append(Document(page_content=caption, metadata=el.metadata))

print(f"Captioned {len(image_docs)} images")

INFO: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Captioned 6 images


In [7]:
print(image_docs[-1].page_content)

- Image type: 2-D line chart (performance curves) with markers, gridlines, and legend.

- Overall subject: comparison of generation accuracy as a function of retrieval accuracy for two methods labeled "Self-RAG" and "Self-CRAG".

- Visible text and labels:
  - Y-axis label (rotated vertically, left side): "Accuracy of generation"
  - X-axis label (bottom): "Accuracy of retrieval"
  - X-axis leftmost tick: "69.8" with "(Actual)" printed beneath it
  - Legend (top-right, boxed): a green star marker with the label "Self-RAG" and a grey diamond marker with the label "Self-CRAG"
  - A horizontal dashed blue line near the bottom annotated in blue text "no retrieval"
  - Y-axis numeric ticks shown from 20 up to 70 (20, 30, 40, 50, 60, 70 visible)

- Plot elements and styles:
  - Two series plotted as connected lines with markers:
    - Self-RAG: green line with green star-shaped markers.
    - Self-CRAG: grey line with grey diamond-shaped markers.
  - Light grey vertical dashed guide lines at

In [8]:
# split text
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

text_elements = [el for el in elements if el.metadata.get("category") != "Image"]

text_docs = splitter.split_documents(text_elements)
caption_docs = splitter.split_documents(image_docs)

all_docs = text_docs + caption_docs
print(f"Total chunks: {len(all_docs)} ({len(text_docs)} text + {len(caption_docs)} captions)")

Total chunks: 279 (258 text + 21 captions)


In [9]:
# embeddings
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [10]:
# vector store
vector_store = Chroma.from_documents(filter_complex_metadata(all_docs), embeddings)

INFO: HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


In [11]:
# retriever
retriever = vector_store.as_retriever(search_kwargs={"k": 4})

In [12]:
# RAG chain
prompt = ChatPromptTemplate.from_messages([
    ("human", "Answer the question based only on the following context:\n\n{context}\n\nQuestion: {question}"),
])

def build_messages(inputs):
    docs = inputs["docs"]
    question = inputs["question"]

    text_chunks = [d for d in docs if d.metadata.get("category") != "Image"]
    image_chunks = [d for d in docs if d.metadata.get("category") == "Image"]

    text_context = "\n\n".join(d.page_content for d in text_chunks)

    messages = prompt.format_messages(context=text_context, question=question)

    if image_chunks:
        seen_paths = set()
        image_content = []
        for doc in image_chunks:
            image_path = doc.metadata.get("image_path")
            if image_path and image_path not in seen_paths:
                seen_paths.add(image_path)
                image_content.append({
                    "type": "image_url",
                    "image_url": {"url": f"data:image/jpeg;base64,{encode_image(image_path)}"},
                })

        if image_content:
            text_content = messages[-1].content
            messages[-1] = HumanMessage(content=[
                {"type": "text", "text": text_content},
                *image_content,
            ])

    return messages

llm = ChatOpenAI(model="gpt-5-mini")

chain = (
    {"docs": retriever, "question": RunnablePassthrough()}
    | RunnableLambda(build_messages)
    | {"response": llm | StrOutputParser(), "context": RunnablePassthrough()}
)

In [13]:
# test
question = "How does Self-CRAG compares with Self-RAG as shown in the line chart. Can you explain this in a little bit more detail?"
answer = chain.invoke(question)
print(answer["response"])

INFO: HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Summary: Self-CRAG (gray diamonds) is consistently better than Self-RAG (green stars) across every retrieval-accuracy point in the chart, and it degrades more slowly as retrieval accuracy drops — i.e., it is more robust to poorer retrieval.

Concrete examples from the plot
- At the actual retrieval accuracy (~69.8): Self-CRAG ≈ 62 vs Self-RAG ≈ 55 (≈7 percentage points higher).
- At retrieval = 50: Self-CRAG ≈ 59 vs Self-RAG ≈ 46 (≈13 pp higher).
- At retrieval = 10: Self-CRAG ≈ 52 vs Self-RAG ≈ 33 (≈19 pp higher).

Other observations
- Both methods lose generation accuracy as retrieval accuracy falls, but Self-RAG’s decline is steeper.
- Even when retrieval is very poor (near the “no retrieval” baseline at ≈28), Self-CRAG stays substantially above that baseline and above Self-RAG, showing it makes better use of (or is less harmed by noisy) retrieved evidence.

Takeaway: Self-CRAG not only raises absolute generation accuracy but also increases robustness to lower-quality retrieval comp

In [14]:
len(answer["context"][0].content)

2

In [15]:
question = "Computational requirements of CRAG vs Self-RAG and which was has faster execution time and can you give me the actual TFLOPS values?"
answer = chain.invoke(question)
print(answer["response"])

INFO: HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


From Table 6 (generation-phase estimates):

- CRAG: ~27.2 TFLOPs per token; average execution time 0.512 s per instance.  
- Self-RAG: range ~26.5 → 132.4 TFLOPs per token (adaptive, depends on input/strategy); average execution time 0.741 s per instance.

So CRAG has lower and fixed computational requirement (~27.2 TFLOPs/token) and is faster (0.512 s) than Self-RAG (0.741 s). Note these are rough FLOPs estimates for the generation phase only (retrieval/data-processing not included).
